# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant Schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Print basic metadata (access as attributes, not like a dictionary)
print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {metadata.date_published}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets and their IDs
print("Record sets in this dataset (by @id):\n")
for record_set in metadata.record_sets:
    print(f"- {record_set['@id']} (name: {record_set.get('name', '<no name>')})")

# For demonstration, review fields and columns in each record set
print("\nExample fields and columns in record sets:")
for record_set in metadata.record_sets:
    print(f"\nRecordSet @id: {record_set['@id']}")
    fields = record_set.get('fields', [])
    for field in fields:
        print(f"  Field @id: {field['@id']}, name: {field.get('name', '<no name>')}, dataType: {field.get('dataType', '<unknown>')}")
        column = field.get('column')
        if column:
            print(f"    Column @id: {column['@id']} (source: {column.get('source', '<unknown>')})")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# We'll extract data from all record sets (by their @id)
record_sets = [rs['@id'] for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} rows from RecordSet: {record_set_id}")

# For the main record set, display columns and preview data
main_record_set_id = record_sets[0] if len(record_sets) else None
if main_record_set_id:
    print("\nColumns in main record set:")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For demonstration, pick a numeric field from the DataFrame
# Let's pick 'Coverage (%)' (if present) for abundance/coverage analysis
import numpy as np

possible_numeric_fields = ['Coverage (%)', 'Peptide Count', 'MW [kDa]', 'Abundance', 'Normalized Abundance']
main_df = dataframes.get(main_record_set_id)

# Find the first existing field in the DataFrame
numeric_field = None
for f in possible_numeric_fields:
    if f in main_df.columns:
        numeric_field = f
        break

if numeric_field is None:
    raise ValueError("Cannot find a suitable numeric field for EDA. Please review the dataframe columns.")

threshold = main_df[numeric_field].mean()
filtered_df = main_df[main_df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): {filtered_df.shape[0]} rows out of {main_df.shape[0]}")

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / (filtered_df[numeric_field].std() + 1e-9)
print(f"Normalized {numeric_field} for filtered records (top 5):")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a categorical field if it exists (e.g., 'Sample', 'Description')
possible_group_fields = ['Sample', 'Description', 'Protein Name']
group_field = None
for gf in possible_group_fields:
    if gf in main_df.columns:
        group_field = gf
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"\nGrouped data by {group_field} (mean values):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(main_df[numeric_field].dropna(), kde=True, bins=30, color='skyblue')
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If grouped, show mean values by the group field
if group_field:
    plt.figure(figsize=(10,4))
    sorted_means = grouped_df[numeric_field].sort_values(ascending=False).head(10)
    sns.barplot(y=sorted_means.index, x=sorted_means.values, palette="viridis")
    plt.title(f"Top 10 '{group_field}' by mean {numeric_field}")
    plt.xlabel(f"Mean {numeric_field}")
    plt.ylabel(group_field)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR^2 mass spectrometry dataset using the `mlcroissant` library. We've demonstrated how to inspect dataset structure by `@id`, extract and load records, and perform basic filtering and normalization of numeric fields. Visual analysis further enabled us to discover trends in protein coverage and key groupings.

To continue, explore other fields, visualize additional relationships, or apply domain-specific analyses based on your research questions.